# evergreen_lab — 전략 평가 노트북

전략을 골라 과거 캔들에 돌려 **얼마 벌었는지**를 본다. 현물(매수=풀, 매도=청산) 기준.

- 전략 추가: `evergreen_lab/strategies/`에 파일 1개 + `strategies/__init__.py`에 import 1줄.
- 실데이터는 `outputs/data/upbit-cache`에 CSV로 캐시된다(최초 실행만 Upbit API 호출). 먼저 `uv sync`.


## 1. 셋업 (프로젝트 경로 + import)

In [ ]:
import os, sys
from pathlib import Path

# 'evergreen_lab'를 담은 프로젝트 루트를 찾아 sys.path에 추가한다.
root = Path.cwd()
while root != root.parent and not (root / 'evergreen_lab').is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import evergreen_lab as lab
print('registered strategies:', lab.list_strategies())


## 2. 여기만 바꿔서 실행

`전략`은 `lab.list_strategies()` 중 하나. **trend2(=v6 룰)는 `minute_240`, v1~v5는 `days`** 캔들을 권장.

In [ ]:
from datetime import datetime, timezone

전략 = 'trend2'                       # 'trend2' | 'v1' | 'v2' | 'v3' | 'v4' | 'v5'
마켓 = 'KRW-BTC'
시작일 = datetime(2020, 1, 1, tzinfo=timezone.utc)
종료일 = None                          # None = 지금까지
interval = 'minute_240'                # trend2=minute_240, v1~v5='days'
비용 = lab.Cost(fee_per_side=0.0005, slippage=0.0002)
전략_파라미터 = {}                      # 예: {'buy_cutoff': 0.08}


## 3. 평가 (수익 요약 + 자산 곡선)

In [ ]:
결과 = lab.evaluate(
    전략, market=마켓, from_dt=시작일, to_dt=종료일,
    interval=interval, cost=비용, **전략_파라미터,
)
print(결과.describe())
결과.plot_equity()


## 4. 전체 전략 비교

각 전략을 권장 interval로 돌려 한 표로 본다. (trend2는 minute_240, 나머지는 days라 기간·표본이 달라 절대 수익 직접 비교는 주의.)

In [ ]:
import pandas as pd

권장_interval = {'trend2': 'minute_240', 'v1': 'days', 'v2': 'days', 'v3': 'days', 'v4': 'days', 'v5': 'days'}
행 = []
for name in ['trend2', 'v1', 'v2', 'v3', 'v4', 'v5']:
    r = lab.evaluate(name, market=마켓, from_dt=시작일, to_dt=종료일,
                     interval=권장_interval[name], cost=비용)
    s = r.summary
    행.append({'strategy': name, 'interval': 권장_interval[name],
               'total_return': s.total_return, 'buy_hold': s.buy_hold_return,
               'cagr': s.cagr, 'mdd': s.mdd, 'round_trips': s.round_trips,
               'win_rate': s.win_rate})
비교표 = pd.DataFrame(행).sort_values('total_return', ascending=False).reset_index(drop=True)
비교표
